In [2]:
import pandas as pd
import requests
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'


In [3]:
years = [2019, 2020, 2021, 2022, 2023]

session = requests.Session()
retries = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=0.75,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10))

def fetch_unique_public_hs_us(year: int) -> int:
    url = f"https://educationdata.urban.org/api/v1/schools/ccd/directory/{year}/"
    ids = set()
    while url:
        resp = session.get(url, timeout=(15, 120))
        resp.raise_for_status()
        data = resp.json()
        for row in data.get("results", []):
            ncessch = row.get("ncessch")
            if ncessch is not None and str(ncessch).strip() != "":
                ids.add(str(ncessch).strip())
        url = data.get("next")
    return len(ids)

In [4]:
merged_geo = pd.read_csv(DATA_DIR / 'merged_geo.csv')
merged_geo['cycle'] = pd.to_numeric(merged_geo['cycle'], errors='coerce').astype('Int64')
merged_geo['hs_id'] = merged_geo['hs_id'].astype('string').str.strip()
merged_geo['school_type'] = merged_geo['school_type'].astype('string').str.lower().str.strip()

pss_2122 = pd.read_csv(DATA_DIR / 'pss_2021-22.csv')
pss_1920 = pd.read_csv(DATA_DIR / 'pss_2019-20.csv')

pss_keep_cols = ['PPIN']
pss_2122_small = pss_2122[pss_keep_cols].copy()
pss_1920_small = pss_1920[pss_keep_cols].copy()

private_us_1920 = pss_1920_small['PPIN'].astype('string').str.strip().dropna()
private_us_2122 = pss_2122_small['PPIN'].astype('string').str.strip().dropna()
private_us_count_1920 = private_us_1920[private_us_1920 != ''].nunique()
private_us_count_2122 = private_us_2122[private_us_2122 != ''].nunique()

/var/folders/wz/pjx159kj141cgdkvkdrlfb040000gn/T/ipykernel_82958/3755675000.py:6: DtypeWarning: Columns (0: PPIN, 1: SLDUST22) have mixed types. Specify dtype option on import or set low_memory=False.
  pss_2122 = pd.read_csv(DATA_DIR / 'pss_2021-22.csv')
/var/folders/wz/pjx159kj141cgdkvkdrlfb040000gn/T/ipykernel_82958/3755675000.py:7: DtypeWarning: Columns (0: SLDLST20, 1: SLDUST20) have mixed types. Specify dtype option on import or set low_memory=False.
  pss_1920 = pd.read_csv(DATA_DIR / 'pss_2019-20.csv')


In [5]:
rows = []
for year in years:
    public_us_count = fetch_unique_public_hs_us(year)
    public_visited = merged_geo.loc[
        (merged_geo['school_type'] == 'public') & (merged_geo['cycle'] == year),
        'hs_id'
    ]
    private_visited = merged_geo.loc[
        (merged_geo['school_type'] == 'private') & (merged_geo['cycle'] == year),
        'hs_id'
    ]

    rows.append({
        'year': year,
        '# of Public High Schools in US': public_us_count,
        '# of Public High Schools Visited': public_visited[public_visited.notna() & (public_visited != '')].nunique(),
        '# of private high schools in US': private_us_count_1920 if year in [2019, 2020] else private_us_count_2122,
        '# of private high schools visited': private_visited[private_visited.notna() & (private_visited != '')].nunique(),
    })

hs_total_vs_visited = pd.DataFrame(rows)
hs_total_vs_visited

KeyboardInterrupt: 

In [7]:
out_path = DATA_DIR / 'hs_total_vs_visited.csv'
hs_total_vs_visited.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
hs_total_vs_visited

Saved: /Users/bryceclement/Desktop/c2i/data/hs_total_vs_visited.csv


,year,# of Public High Schools in US,# of Public High Schools Visited,# of private high schools in US,# of private high schools visited
0,2019,101688,61903,21572,12701
1,2020,101662,50323,21572,11289
2,2021,102130,56266,22345,11887
3,2022,102268,59815,22345,12532
4,2023,102274,51615,22345,11487


In [6]:
# Output CCD unique-school counts for high-school filters (not added to hs_total_vs_visited)
level_targets = {3, 7}
grade_targets = {10, 11, 12, 13}

for year in years:
    url = f"https://educationdata.urban.org/api/v1/schools/ccd/directory/{year}/"
    ids_level = set()
    ids_grade = set()

    while url:
        resp = session.get(url, timeout=(15, 120))
        resp.raise_for_status()
        data = resp.json()

        for row in data.get('results', []):
            ncessch = row.get('ncessch')
            if ncessch is None or str(ncessch).strip() == '':
                continue
            school_id = str(ncessch).strip()

            school_level = pd.to_numeric(row.get('school_level'), errors='coerce')
            highest_grade = pd.to_numeric(row.get('highest_grade_offered'), errors='coerce')

            if pd.notna(school_level) and int(school_level) in level_targets:
                ids_level.add(school_id)

            if pd.notna(highest_grade) and int(highest_grade) in grade_targets:
                ids_grade.add(school_id)

        url = data.get('next')

    print(f"{year} | unique ncessch where school_level in [3, 7]: {len(ids_level):,}")
    print(f"{year} | unique ncessch where highest_grade_offered in [10, 11, 12, 13]: {len(ids_grade):,}")


2019 | unique ncessch where school_level in [3, 7]: 23,789
2019 | unique ncessch where highest_grade_offered in [10, 11, 12, 13]: 27,681
2020 | unique ncessch where school_level in [3, 7]: 23,779
2020 | unique ncessch where highest_grade_offered in [10, 11, 12, 13]: 27,725
2021 | unique ncessch where school_level in [3, 7]: 24,017
2021 | unique ncessch where highest_grade_offered in [10, 11, 12, 13]: 28,264
2022 | unique ncessch where school_level in [3, 7]: 24,070
2022 | unique ncessch where highest_grade_offered in [10, 11, 12, 13]: 28,342
2023 | unique ncessch where school_level in [3, 7]: 24,207
2023 | unique ncessch where highest_grade_offered in [10, 11, 12, 13]: 28,523


In [7]:
# Build filtered high-school counts (US + visited), merge CCD attrs, and save final_data.csv
level_targets = {3, 7}

ccd_public_us_counts = {}
ccd_attr_frames = []

for year in years:
    url = f"https://educationdata.urban.org/api/v1/schools/ccd/directory/{year}/"
    ids_public_high = set()
    rows = []

    while url:
        resp = session.get(url, timeout=(15, 120))
        resp.raise_for_status()
        data = resp.json()

        for row in data.get('results', []):
            ncessch = row.get('ncessch')
            if ncessch is None or str(ncessch).strip() == '':
                continue

            hs_id = str(ncessch).strip()
            school_level = pd.to_numeric(row.get('school_level'), errors='coerce')
            highest_grade = pd.to_numeric(row.get('highest_grade_offered'), errors='coerce')

            rows.append({
                'cycle': year,
                'hs_id': hs_id,
                'hs_school_level': int(school_level) if pd.notna(school_level) else pd.NA,
                'hs_highest_grade_offered': int(highest_grade) if pd.notna(highest_grade) else pd.NA,
            })

            if pd.notna(school_level) and int(school_level) in level_targets:
                ids_public_high.add(hs_id)

        url = data.get('next')

    ccd_public_us_counts[year] = len(ids_public_high)
    ccd_year_df = pd.DataFrame(rows).drop_duplicates(subset=['cycle', 'hs_id'])
    ccd_attr_frames.append(ccd_year_df)

ccd_attr = pd.concat(ccd_attr_frames, ignore_index=True)
ccd_attr['cycle'] = pd.to_numeric(ccd_attr['cycle'], errors='coerce').astype('Int64')
ccd_attr['hs_id'] = ccd_attr['hs_id'].astype('string').str.strip()
ccd_attr['hs_school_level'] = pd.to_numeric(ccd_attr['hs_school_level'], errors='coerce').astype('Int64')
ccd_attr['hs_highest_grade_offered'] = pd.to_numeric(ccd_attr['hs_highest_grade_offered'], errors='coerce').astype('Int64')

# Private US counts from PSS where LEVEL in [2, 3]
private_us_counts_level23 = {}
for year in years:
    src = pss_1920 if year in [2019, 2020] else pss_2122
    tmp = src[['PPIN', 'LEVEL']].copy()
    tmp['PPIN'] = tmp['PPIN'].astype('string').str.strip()
    tmp['LEVEL'] = pd.to_numeric(tmp['LEVEL'], errors='coerce').astype('Int64')
    tmp = tmp[tmp['LEVEL'].isin([2, 3])]
    private_us_counts_level23[year] = tmp.loc[tmp['PPIN'].notna() & (tmp['PPIN'] != ''), 'PPIN'].nunique()

# Load updated_data and add 2 CCD columns
updated_data = pd.read_csv(DATA_DIR / 'updated_data.csv')
updated_data['cycle'] = pd.to_numeric(updated_data['cycle'], errors='coerce').astype('Int64')
updated_data['hs_id'] = updated_data['hs_id'].astype('string').str.strip()
updated_data['school_type'] = updated_data['school_type'].astype('string').str.lower().str.strip()

final_data = updated_data.merge(
    ccd_attr[['cycle', 'hs_id', 'hs_school_level', 'hs_highest_grade_offered']],
    on=['cycle', 'hs_id'],
    how='left'
)

final_out = DATA_DIR / 'final_data.csv'
final_data.to_csv(final_out, index=False)
print(f"Saved: {final_out}")

# If private level is not present in updated_data, create a temporary per-year lookup for visited counts
if 'hs_priv_level' not in final_data.columns:
    pss1920_level = pss_1920[['PPIN', 'LEVEL']].copy()
    pss2122_level = pss_2122[['PPIN', 'LEVEL']].copy()

    pss1920_level = pss1920_level.rename(columns={'PPIN': 'hs_id', 'LEVEL': 'hs_priv_level'})
    pss2122_level = pss2122_level.rename(columns={'PPIN': 'hs_id', 'LEVEL': 'hs_priv_level'})

    pss1920_level['hs_id'] = pss1920_level['hs_id'].astype('string').str.strip()
    pss2122_level['hs_id'] = pss2122_level['hs_id'].astype('string').str.strip()
    pss1920_level['hs_priv_level'] = pd.to_numeric(pss1920_level['hs_priv_level'], errors='coerce').astype('Int64')
    pss2122_level['hs_priv_level'] = pd.to_numeric(pss2122_level['hs_priv_level'], errors='coerce').astype('Int64')

    pss_level_by_year = []
    for year in [2019, 2020]:
        tmp = pss1920_level.copy()
        tmp['cycle'] = year
        pss_level_by_year.append(tmp)
    for year in [2021, 2022, 2023]:
        tmp = pss2122_level.copy()
        tmp['cycle'] = year
        pss_level_by_year.append(tmp)

    pss_level_by_year = pd.concat(pss_level_by_year, ignore_index=True).drop_duplicates(['cycle', 'hs_id'])
    pss_level_by_year['cycle'] = pd.to_numeric(pss_level_by_year['cycle'], errors='coerce').astype('Int64')

    private_level_for_counts = final_data[['cycle', 'hs_id', 'school_type']].merge(
        pss_level_by_year,
        on=['cycle', 'hs_id'],
        how='left'
    )
else:
    private_level_for_counts = final_data[['cycle', 'hs_id', 'school_type', 'hs_priv_level']].copy()
    private_level_for_counts['hs_priv_level'] = pd.to_numeric(private_level_for_counts['hs_priv_level'], errors='coerce').astype('Int64')

# Store year-level visited counts based on the requested filters
public_visited_high = {}
private_visited_level23 = {}

for year in years:
    public_mask = (
        (final_data['school_type'] == 'public')
        & (final_data['cycle'] == year)
        & (final_data['hs_school_level'].isin([3, 7]))
    )
    public_visited_high[year] = final_data.loc[public_mask, 'hs_id'].dropna().nunique()

    private_mask = (
        (private_level_for_counts['school_type'] == 'private')
        & (private_level_for_counts['cycle'] == year)
        & (private_level_for_counts['hs_priv_level'].isin([2, 3]))
    )
    private_visited_level23[year] = private_level_for_counts.loc[private_mask, 'hs_id'].dropna().nunique()

print('Public US counts (school_level in [3,7]):')
print(pd.Series(ccd_public_us_counts).rename_axis('year').to_frame('count'))
print('\nPrivate US counts (LEVEL in [2,3]):')
print(pd.Series(private_us_counts_level23).rename_axis('year').to_frame('count'))
print('\nVisited counts (filtered):')
print(pd.DataFrame({
    'year': years,
    'public_visited_level_3_7': [public_visited_high[y] for y in years],
    'private_visited_level_2_3': [private_visited_level23[y] for y in years],
}))


Saved: /Users/bryceclement/Desktop/c2i/data/final_data.csv
Public US counts (school_level in [3,7]):
      count
year       
2019  23789
2020  23779
2021  24017
2022  24070
2023  24207

Private US counts (LEVEL in [2,3]):
      count
year       
2019   8168
2020   8168
2021   8764
2022   8764
2023   8764

Visited counts (filtered):
   year  public_visited_level_3_7  private_visited_level_2_3
0  2019                     15968                       4695
1  2020                     13249                       4234
2  2021                     14914                       4449
3  2022                     15406                       4712
4  2023                     13392                       4277


In [8]:
# Update total-vs-visited table with public level [3,7] and private level [2,3]
rows_filtered = []
for year in years:
    rows_filtered.append({
        'year': year,
        '# of Public High Schools in US': ccd_public_us_counts[year],
        '# of Public High Schools Visited': public_visited_high[year],
        '# of private high schools in US': private_us_counts_level23[year],
        '# of private high schools visited': private_visited_level23[year],
    })

hs_total_vs_visited = pd.DataFrame(rows_filtered)
hs_total_vs_visited.to_csv(DATA_DIR / 'hs_total_vs_visited.csv', index=False)
hs_total_vs_visited


,year,# of Public High Schools in US,# of Public High Schools Visited,# of private high schools in US,# of private high schools visited
0,2019,23789,15968,8168,4695
1,2020,23779,13249,8168,4234
2,2021,24017,14914,8764,4449
3,2022,24070,15406,8764,4712
4,2023,24207,13392,8764,4277
